In [0]:
%sql
SHOW CATALOGS;
--SHOW VOLUMES; --default schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.df_project1;

In [0]:
%sql
SHOW VOLUMES IN workspace.df_project1;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.df_project1.data;


In [0]:
from pyspark.sql.types import *
arqschema = "id INT, nome STRING, status STRING, cidade STRING, vendas INT, data STRING"

despachantes = spark.read.csv("/Volumes/workspace/df_project1/data/despachantes.csv", header=False, schema=arqschema)
despachantes.show()

In [0]:
desp_autoschema = spark.read.load("/Volumes/workspace/df_project1/data/despachantes.csv",
                     format="csv", sep=",", inferSchema=True, header=False)
desp_autoschema.show()

In [0]:
desp_autoschema.printSchema()
despachantes.printSchema()


In [0]:
from pyspark.sql import functions as Func
despachantes.select("id","nome","vendas").where(Func.col("vendas") > 20).show()
despachantes.select("id","nome","vendas").where((Func.col("vendas") > 20) & (Func.col("vendas") < 40)).show()

In [0]:
novodf = despachantes.withColumnRenamed("nome","nomes")
novodf.columns

In [0]:
from pyspark.sql.functions import *
despachantes2 = despachantes.withColumn("data2", to_timestamp(Func.col("data"),"yyyy-MM-dd"))
despachantes2.printSchema()

In [0]:
despachantes2.select(year("data")).show()
despachantes2.select(year("data")).distinct().show()
despachantes2.select("nome",year("data")).orderBy("nome").show()
despachantes2.select("data").groupBy(year("data")).count().show()
despachantes2.select(Func.sum("vendas")).show()

In [0]:
despachantes.write.format("parquet").save("/Volumes/workspace/df_project1/data/dfimportparquet")
despachantes.write.format("csv").save("/Volumes/workspace/df_project1/data/dfimportcsv")
despachantes.write.format("json").save("/Volumes/workspace/df_project1/data/dfimportjson")
despachantes.write.format("orc").save("/Volumes/workspace/df_project1/data/dfimportorc")

In [0]:
par = spark.read.format("parquet").load("/Volumes/workspace/df_project1/data/dfimportparquet/*.parquet")
par.show()
par.printSchema()

In [0]:
despachantes.show(1)

despachantes.take(2)

In [0]:
despachantes.collect()

In [0]:
despachantes.count()

In [0]:
despachantes.orderBy("vendas").show()
despachantes.orderBy(Func.col("vendas").desc()).show()

In [0]:
despachantes.orderBy(Func.col("cidade").desc(),Func.col("vendas").desc()).show()

In [0]:
despachantes.groupBy("cidade").agg(sum("vendas")).show()